# YuE2 · качество записи на WildSongBench

In [ ]:
import hashlib
import json
import os
import zipfile
from pathlib import Path

project = "yue2"
archive_name = "replaceme-yue2-env.zip"
expected_manifest = "ed17d5990dfebed1e88f396676445c0d7f5109bb1f698877d14e28b9a42ece2e"
kaggle = Path("/kaggle/input").is_dir()
default_input = Path("/kaggle/input/replaceme-yue2-env") if kaggle else Path.cwd()
inputs = Path(os.environ.get("REPLACEME_INPUT", default_input)).expanduser().resolve()
work = Path(os.environ.get("REPLACEME_WORK", "/kaggle/working" if kaggle else "replaceme-work")).resolve()

archive_path = inputs if inputs.name == archive_name else inputs / archive_name
if archive_path.is_file():
    bundle = archive_path
    with zipfile.ZipFile(bundle) as archive:
        metadata = archive.read("environment.json")
        bootstrap = archive.read("bootstrap.py")
elif inputs.is_dir() and (inputs / "environment.json").is_file():
    bundle = inputs
    metadata = (bundle / "environment.json").read_bytes()
    bootstrap = (bundle / "bootstrap.py").read_bytes()
else:
    raise FileNotFoundError(f"Ожидается {archive_path}. Укажите точный путь к ZIP или каталогу Dataset в REPLACEME_INPUT.")

if hashlib.sha256(metadata).hexdigest() != expected_manifest:
    raise ValueError(f"Манифест не соответствует комплекту {archive_name}.")
manifest = json.loads(metadata)
if manifest["project"] != project or hashlib.sha256(bootstrap).hexdigest() != manifest["bootstrap_sha256"]:
    raise ValueError("Ноутбук и загрузчик окружения относятся к разным сборкам.")
scope = {}
exec(compile(bootstrap, "bootstrap.py", "exec"), scope)
scope["start"](bundle, manifest, work / project)


## Промпты и бюджет генерации

In [ ]:
%%benchmark
import json, os, runpy, sys
from datetime import datetime, timezone
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import soundfile as sf
import torch
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from yue2 import YuE2Pipeline
from yue2.protocol import SongRequest
from yue2.storage import verify_result

map_checkpoint = runpy.run_path(str(Path(sys.prefix) / "share/yue2/yue2_model.py"))["map_checkpoint"]

def checkpoint(repo, revision, extra=()):
    for name in ("config.json", "weights_manifest.json", *extra, "model.safetensors"):
        file = hf_hub_download(repo, name, revision=revision)
    return Path(file).parent

model = checkpoint("m-a-p/YuE2-3B", "1a96eca688d6ae5d7f0feb88573fec89920fcd19", ("qwen.tiktoken",))
vae = checkpoint("m-a-p/YuE2-Vae-legacy", "b54118f0fc462f08999d1ec07e88817f4ee3f770")
audiobox = hf_hub_download("facebook/audiobox-aesthetics", "checkpoint.pt",
                           revision="9b1dd8e5df9af7216e836a98974fe3b82c56ded6")
wsb = dict(repo_id="m-a-p/WildSongBench", repo_type="dataset", revision="e361b85a8d7365d3079fa9647641a87ce3c8c5b5")
manifest_file = Path(hf_hub_download(filename="benchmark/reproduction_manifest.jsonl", **wsb))
bench_csv = Path(hf_hub_download(filename="benchmark/benchmark-results.csv", **wsb))
manifest = [json.loads(line) for line in manifest_file.read_text().splitlines()]
indices, abc_limit, audio_limit = [0, 96], 256, 250
rows = [manifest[i] for i in indices]
torch.set_num_threads(min(4, len(os.sched_getaffinity(0))))
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.get_device_name() if device == "cuda" else "CPU")
for row in rows:
    print(row["prompt_index"], row["candidate_ar_seeds"][0], row["style"])

## Генерация

In [ ]:
%%benchmark
root = Path.cwd() / "yue2-results"
root.mkdir(exist_ok=True)
run = Path(mkdtemp(dir=root, prefix=datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-")))
songs = []
with YuE2Pipeline.from_pretrained(model, vae=vae,
                                  device=device, backend="torch-eager", low_vram=True,
                                  memory_budget_gib=8, vae_core_frames=64, progress=False) as pipe:
    map_checkpoint(pipe)
    for i, row in zip(indices, rows):
        seed = row["candidate_ar_seeds"][0]
        assert i == row["prompt_index"] and seed == row["candidate_nar_seeds"][0] == 831001 + i
        request = SongRequest(style=row["style"], lyrics=row["lyrics"], seed=seed, cot="full",
                              cfg_scale=1.0, id=f"wsb_{i:03d}_r0_s{seed}")
        assert request.text() == row["generation_prompt"]
        print(f"Генерация {request.id}…", flush=True)
        song = pipe(**request.to_dict(), abc_sampling={"max_tokens": abc_limit},
                    semantic_sampling={"max_tokens": audio_limit})
        path = run / request.id
        song.save_artifacts(path)
        receipt = verify_result(path, song.request_identity)
        songs.append((i, seed, path, receipt))
        print(f'{receipt["audio_seconds"]:.2f} с · {path.name}', flush=True)
        del song

## AudioBox Production Quality ↑

In [ ]:
%%benchmark
from audiobox_aesthetics.infer import initialize_predictor
from importlib.metadata import version

predictor = initialize_predictor(audiobox)
scores = []
for i, seed, path, receipt in songs:
    wav, sr = sf.read(path / "audio.flac", dtype="float32", always_2d=True)
    assert sr == 48000 and wav.shape[1] == 2 and np.isfinite(wav).all() and np.any(wav)
    pq = predictor.forward([{"path": torch.from_numpy(wav.T.copy()), "sample_rate": sr}])[0]["PQ"]
    assert np.isfinite(pq)
    scores.append(dict(prompt_index=i, seed=seed, seconds=len(wav)/sr, pq=pq,
                       audio=f"{path.name}/audio.flac", truncated=receipt["truncated"]))
    print(f"WSB {i:03d}   {len(wav)/sr:5.2f} с   PQ {pq:.4f}")

mean = float(np.mean([row["pq"] for row in scores]))
report = dict(rows=scores, mean_pq=mean, paper_pq=8.2598, paper_n=192, device=device,
              generator=version("yue2-infer"), evaluator=version("audiobox-aesthetics"),
              torch=torch.__version__, abc_limit=abc_limit, semantic_limit=audio_limit)
(run / "scores.json").write_text(json.dumps(report, indent=2) + "\n")
print(f"Среднее по {len(scores)} фрагментам   {mean:.4f}")

## Прослушать записи

In [ ]:
%%benchmark
for row in scores:
    samples, rate = sf.read(run / row["audio"], dtype="float32", always_2d=True)
    print(f'WSB {row["prompt_index"]:03d} · PQ {row["pq"]:.4f}')
    display(Audio(samples.T, rate=rate, normalize=False))

## Индекс музыкальности

In [ ]:
%%benchmark
import csv

with bench_csv.open() as stream:
    systems = list(csv.DictReader(stream))
names = [r["system"] for r in systems]
values = np.array([[float(r[k]) for k in ("songbench_musicality", "songeval_musicality")] for r in systems])
reference = values[[not name.startswith("YuE2") for name in names]]
assert len(reference) == 15 and len(systems) == 17 and all(int(r["n"]) == 192 for r in systems)
z = (values - reference.mean(axis=0)) / reference.std(axis=0)
raw = (2 * z[:, 0] + z[:, 1]) / 3
index = 10 + 80 * (raw - raw.min()) / np.ptp(raw)
value = float(index[names.index("YuE2")])
print(f"YuE2   {value:.2f}   ·   статья 81.63")
(run / "figure11.json").write_text(json.dumps({"yue2": value, "paper": 81.63}, indent=2) + "\n")
assert round(value, 2) == 81.63